In [0]:
file_path = "/Volumes/week4_databricks/default/orders/orders (1).csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

display(df)

order_id,customer_id,customer_name,city,amount,status
1001,C001,Aarav,Chennai,750,Completed
1002,C002,Diya,Bengaluru,450,Pending
1003,C003,Rahul,Hyderabad,980,Completed
1004,C004,Sneha,Mumbai,1200,Completed
1005,C005,Kiran,Delhi,300,Cancelled
1006,C006,Ananya,Pune,640,Completed
1007,C007,Vikram,Kochi,890,Pending
1008,C008,Meera,Coimbatore,1500,Completed
1009,C009,Arjun,Madurai,520,Completed
1010,C010,Nisha,Trichy,410,Pending


In [0]:
df = df.dropDuplicates()
df = df.dropna()

display(df)

order_id,customer_id,customer_name,city,amount,status
1006,C006,Ananya,Pune,640,Completed
1010,C010,Nisha,Trichy,410,Pending
1009,C009,Arjun,Madurai,520,Completed
1007,C007,Vikram,Kochi,890,Pending
1002,C002,Diya,Bengaluru,450,Pending
1008,C008,Meera,Coimbatore,1500,Completed
1004,C004,Sneha,Mumbai,1200,Completed
1005,C005,Kiran,Delhi,300,Cancelled
1003,C003,Rahul,Hyderabad,980,Completed
1001,C001,Aarav,Chennai,750,Completed


In [0]:
from pyspark.sql.functions import when, col

updated_df = df.withColumn(
    "status",
    when(col("status") == "Pending", "Delivered")
    .otherwise(col("status"))
)

display(updated_df)

order_id,customer_id,customer_name,city,amount,status
1006,C006,Ananya,Pune,640,Completed
1010,C010,Nisha,Trichy,410,Delivered
1009,C009,Arjun,Madurai,520,Completed
1007,C007,Vikram,Kochi,890,Delivered
1002,C002,Diya,Bengaluru,450,Delivered
1008,C008,Meera,Coimbatore,1500,Completed
1004,C004,Sneha,Mumbai,1200,Completed
1005,C005,Kiran,Delhi,300,Cancelled
1003,C003,Rahul,Hyderabad,980,Completed
1001,C001,Aarav,Chennai,750,Completed


In [0]:
updated_df.write \
.mode("overwrite") \
.format("delta") \
.save("/Volumes/week4_databricks/default/orders/delta_orders")

In [0]:
updated_df.write \
.mode("overwrite") \
.option("header", True) \
.csv("/Volumes/week4_databricks/default/orders/output_csv")

In [0]:
updated_df.createOrReplaceTempView("orders")

In [0]:
%sql
SELECT * FROM orders;

order_id,customer_id,customer_name,city,amount,status
1006,C006,Ananya,Pune,640,Completed
1010,C010,Nisha,Trichy,410,Delivered
1009,C009,Arjun,Madurai,520,Completed
1007,C007,Vikram,Kochi,890,Delivered
1002,C002,Diya,Bengaluru,450,Delivered
1008,C008,Meera,Coimbatore,1500,Completed
1004,C004,Sneha,Mumbai,1200,Completed
1005,C005,Kiran,Delhi,300,Cancelled
1003,C003,Rahul,Hyderabad,980,Completed
1001,C001,Aarav,Chennai,750,Completed


In [0]:
%sql
SELECT COUNT(*) AS TotalOrders
FROM orders;

TotalOrders
10


In [0]:
%sql
SELECT COUNT(*) AS DeliveredOrders
FROM orders
WHERE status='Delivered';

DeliveredOrders
3


In [0]:
%sql
SELECT COUNT(*) AS PendingOrders
FROM orders
WHERE status='Pending';

PendingOrders
0


In [0]:
%sql
SELECT customer_id,
       customer_name,
       COUNT(*) AS delayed_orders
FROM orders
WHERE status='Pending'
GROUP BY customer_id, customer_name
ORDER BY delayed_orders DESC
LIMIT 5;

customer_id,customer_name,delayed_orders


In [0]:
df.createOrReplaceTempView("raw_orders")

In [0]:
%sql
SELECT customer_id,
       customer_name,
       COUNT(*) AS delayed_orders
FROM raw_orders
WHERE status='Pending'
GROUP BY customer_id, customer_name
ORDER BY delayed_orders DESC
LIMIT 5;

customer_id,customer_name,delayed_orders
C010,Nisha,1
C007,Vikram,1
C002,Diya,1
